# Forex Data Analysis

This notebook reads EUR/USD exchange rate data and displays it in a table format.

In [250]:
import pandas as pd
import numpy as np

# Read the EUR/USD data from CSV file
df = pd.read_csv('data/eurusd.csv')

# Display basic information about the data
print(f"Trades: {df.shape[0]}")
print(f"Columns: {df.columns.tolist()}")

Trades: 933
Columns: ['Date', 'Trade', 'Direction', 'EMA', 'SL', 'Pullback', 'TP', 'BOS/CH']


In [251]:
# Display the data table for debugging reasons
df

,Date,Trade,Direction,EMA,SL,Pullback,TP,BOS/CH
0,2025-02-03,#1,Sell,Sell,10.7,10.7,0,BOS
1,2025-02-03,#2,Buy,Buy,5.4,5.4,0,BOS
2,2025-02-03,#4,Buy,Buy,19.3,1.7,24,BOS
3,2025-02-04,#1,Buy,Buy,6.7,0.5,70,BOS
4,2025-02-04,#2,Buy,Sell,7.6,7.6,0,BOS
...,...,...,...,...,...,...,...,...
928,2025-08-08,#2,Sell,Sell,4.0,4.0,0,BOS
929,2025-08-08,#3,Buy,Sell,6.3,6.3,0,CH
930,2025-08-08,#4,Sell,Buy,1.8,1.8,0,CH
931,2025-08-08,#5,Sell,Buy,5.1,5.1,0,CH


In [252]:
# Define RRR calculation function
def calculate_rrr_stats(data_df, strategy_name):
    """
    Calculate Risk-Reward Ratio statistics for a given dataframe and strategy.
    
    Args:
        data_df: DataFrame containing trading data
        strategy_name: Name of the strategy for labeling
    
    Returns:
        DataFrame with RRR statistics
    """
    total_trades = len(data_df)
    
    if total_trades == 0:
        return pd.DataFrame({
            strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome'],
            '1:1 RRR': [0, 0, 0, '0.0%', '0.0%', '0R']
        })
    
    rrr_configs = [
        (1, 50.0),   # 1:1 RRR needs 50% win rate to break even
        (2, 33.0),   # 1:2 RRR needs 33% win rate to break even
        (3, 25.0),   # 1:3 RRR needs 25% win rate to break even
        (4, 20.0),   # 1:4 RRR needs 20% win rate to break even
        (5, 17.0),   # 1:5 RRR needs 17% win rate to break even
        (10, 9.0),   # 1:10 RRR needs 9% win rate to break even
    ]
    
    summary_data = {
        strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
    }
    
    for ratio, breakeven_rate in rrr_configs:
        profitable = data_df[data_df['TP'] > ratio * data_df['SL']]
        wins = len(profitable)
        losses = total_trades - wins
        win_rate = (wins / total_trades) * 100
        edge = win_rate - breakeven_rate
        outcome = (wins * ratio) - losses
        
        summary_data[f'1:{ratio} RRR'] = [
            total_trades,
            wins,
            losses,
            f"{win_rate:.1f}%",
            f"{edge:.1f}%",
            f"{outcome}R"
        ]
    
    return pd.DataFrame(summary_data)

# Define strategy configurations
class Strategy:
    def __init__(self, name, filter_func, description=""):
        self.name = name
        self.filter_func = filter_func
        self.description = description
    
    def apply(self, df):
        """Apply the strategy filter to the dataframe."""
        return self.filter_func(df)

## Strategy Backtesting System

This code provides a modular system for backtesting forex strategies:

### Key Features:
1. **Reusable RRR Calculator**: A single function calculates all Risk-Reward Ratios (1:1 to 1:10)
2. **Strategy Class**: Simple strategy definition with name, filter function, and description
3. **Easy Extension**: Add new strategies by simply appending to the strategies list
4. **No Code Duplication**: Each strategy uses the same calculation logic
5. **Dynamic Display**: All strategies are automatically calculated and displayed

### How to Add New Strategies:
Simply add a new Strategy object with:
- **Name**: Display name for the strategy
- **Filter Function**: Lambda function that filters the dataframe
- **Description**: Optional description of the strategy logic

Example:
```python
strategies.append(
    Strategy(
        "My New Strategy",
        lambda df: df[df['SL'] < 10],  # Your filter logic here
        "Description of what this strategy does"
    )
)
```

In [253]:
# Define all strategies using the new system
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "All trades without any filtering"
    ),
    Strategy(
        "1pip Pullback Strategy",
        lambda df: df[df['Pullback'] >= 1],
        "Trades with at least 1 pip pullback"
    ),
    Strategy(
        "BOS Strategy",
        lambda df: df[df['BOS/CH'] == 'BOS'],
        "Trades with Break of Structure"
    ),
    Strategy(
        "EMA Strategy",
        lambda df: df[df['EMA'] == df['Direction']],
        "Trades where EMA aligns with trade direction"
    ),
    Strategy(
        "EMA + BOS Strategy",
        lambda df: df[(df['EMA'] == df['Direction']) | (df['BOS/CH'] == 'BOS')],
        "Trades with either EMA alignment or BOS"
    ),
]

In [254]:
# Advanced: Programmatically generate strategy combinations
def create_pullback_strategies(pullback_values=[1, 5, 10, 20]):
    """Generate multiple pullback strategies with different thresholds."""
    pullback_strategies = []
    for value in pullback_values:
        pullback_strategies.append(
            Strategy(
                f"{value}pip Pullback",
                lambda df, v=value: df[df['Pullback'] >= v],
                f"Trades with at least {value} pip pullback"
            )
        )
    return pullback_strategies

# Uncomment to add multiple pullback strategies at once:
strategies.extend(create_pullback_strategies([1, 2, 3, 4, 5, 10, 15]))

# Advanced: Create strategy analyzer
def analyze_strategy_correlation(strategy_results):
    """Analyze which strategies perform best at different RRR levels."""
    best_strategies = {}
    for rrr in ['1:1 RRR', '1:2 RRR', '1:3 RRR', '1:4 RRR', '1:5 RRR', '1:10 RRR']:
        best_outcome = -float('inf')
        best_strategy = None
        
        for strategy_name, df in strategy_results.items():
            # Extract outcome value (remove 'R' suffix and convert to int)
            outcome_str = df[rrr].iloc[5]  # Row 5 is 'Outcome'
            outcome = int(outcome_str.replace('R', ''))
            
            if outcome > best_outcome:
                best_outcome = outcome
                best_strategy = strategy_name
        
        best_strategies[rrr] = (best_strategy, best_outcome)
    
    return best_strategies

# Uncomment to see which strategy performs best at each RRR:
# best_performers = analyze_strategy_correlation(strategy_results)
# print("\nBest performing strategies by RRR:")
# for rrr, (strategy, outcome) in best_performers.items():
#     print(f"{rrr}: {strategy} with {outcome}R outcome")

In [255]:
# Calculate all strategy results
strategy_results = {}
for strategy in strategies:
    filtered_df = strategy.apply(df)
    summary_df = calculate_rrr_stats(filtered_df, strategy.name)
    strategy_results[strategy.name] = summary_df

# Display all strategy results
for strategy_name, summary_df in strategy_results.items():
    # Style the first column to have a fixed width
    styled_df = summary_df.style.set_properties(
        subset=[strategy_name], 
        **{'width': '150px'}
    )
    display(styled_df)
    print('')  # Add spacing between tables

,Plain Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,933,933,933,933,933,933
1,Wins,407,310,247,182,147,30
2,Losses,526,623,686,751,786,903
3,Win Rate,43.6%,33.2%,26.5%,19.5%,15.8%,3.2%
4,Edge,-6.4%,0.2%,1.5%,-0.5%,-1.2%,-5.8%
5,Outcome,-119R,-3R,55R,-23R,-51R,-603R


,1pip Pullback Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,824,824,824,824,824,824
1,Wins,307,230,182,136,108,19
2,Losses,517,594,642,688,716,805
3,Win Rate,37.3%,27.9%,22.1%,16.5%,13.1%,2.3%
4,Edge,-12.7%,-5.1%,-2.9%,-3.5%,-3.9%,-6.7%
5,Outcome,-210R,-134R,-96R,-144R,-176R,-615R


,BOS Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,651,651,651,651,651,651
1,Wins,279,217,172,129,102,23
2,Losses,372,434,479,522,549,628
3,Win Rate,42.9%,33.3%,26.4%,19.8%,15.7%,3.5%
4,Edge,-7.1%,0.3%,1.4%,-0.2%,-1.3%,-5.5%
5,Outcome,-93R,0R,37R,-6R,-39R,-398R


,EMA Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,600,600,600,600,600,600
1,Wins,270,207,165,119,96,21
2,Losses,330,393,435,481,504,579
3,Win Rate,45.0%,34.5%,27.5%,19.8%,16.0%,3.5%
4,Edge,-5.0%,1.5%,2.5%,-0.2%,-1.0%,-5.5%
5,Outcome,-60R,21R,60R,-5R,-24R,-369R


,EMA + BOS Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,743,743,743,743,743,743
1,Wins,321,248,196,146,118,25
2,Losses,422,495,547,597,625,718
3,Win Rate,43.2%,33.4%,26.4%,19.7%,15.9%,3.4%
4,Edge,-6.8%,0.4%,1.4%,-0.3%,-1.1%,-5.6%
5,Outcome,-101R,1R,41R,-13R,-35R,-468R


...

,3pip Pullback,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,584,584,584,584,584,584
1,Wins,164,115,86,66,47,3
2,Losses,420,469,498,518,537,581
3,Win Rate,28.1%,19.7%,14.7%,11.3%,8.0%,0.5%
4,Edge,-21.9%,-13.3%,-10.3%,-8.7%,-9.0%,-8.5%
5,Outcome,-256R,-239R,-240R,-254R,-302R,-551R


,4pip Pullback,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,434,434,434,434,434,434
1,Wins,111,70,53,42,31,2
2,Losses,323,364,381,392,403,432
3,Win Rate,25.6%,16.1%,12.2%,9.7%,7.1%,0.5%
4,Edge,-24.4%,-16.9%,-12.8%,-10.3%,-9.9%,-8.5%
5,Outcome,-212R,-224R,-222R,-224R,-248R,-412R


,5pip Pullback,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,305,305,305,305,305,305
1,Wins,70,44,33,25,18,1
2,Losses,235,261,272,280,287,304
3,Win Rate,23.0%,14.4%,10.8%,8.2%,5.9%,0.3%
4,Edge,-27.0%,-18.6%,-14.2%,-11.8%,-11.1%,-8.7%
5,Outcome,-165R,-173R,-173R,-180R,-197R,-294R


,10pip Pullback,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,63,63,63,63,63,63
1,Wins,7,6,4,3,1,0
2,Losses,56,57,59,60,62,63
3,Win Rate,11.1%,9.5%,6.3%,4.8%,1.6%,0.0%
4,Edge,-38.9%,-23.5%,-18.7%,-15.2%,-15.4%,-9.0%
5,Outcome,-49R,-45R,-47R,-48R,-57R,-63R


,15pip Pullback,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,17,17,17,17,17,17
1,Wins,2,1,1,1,0,0
2,Losses,15,16,16,16,17,17
3,Win Rate,11.8%,5.9%,5.9%,5.9%,0.0%,0.0%
4,Edge,-38.2%,-27.1%,-19.1%,-14.1%,-17.0%,-9.0%
5,Outcome,-13R,-14R,-13R,-12R,-17R,-17R
